# Practice Lab: Understanding AI Application Costs (Lesson 13.1)

Module 13 . 4 exercises . Cost calculator + BigQuery logging + budget alerts + anomaly detection


# Lesson 13.1: Understanding AI Application Costs

4 exercises: 1E + 2M + 1C

Module 13: LLM Optimization & Self-Hosting


In [ ]:
import random, json, uuid
from dataclasses import dataclass
from datetime import datetime, timedelta
from collections import defaultdict
random.seed(42)


---
## Exercise 1 (Easy): Cost Calculator


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
from dataclasses import dataclass

@dataclass
class MP:
    provider:str; model:str; input_per_1m:float; output_per_1m:float; category:str

DB=[MP("google","gemini-2.0-flash",0.10,0.40,"fast"),MP("google","gemini-2.5-flash",0.15,0.60,"fast"),
    MP("google","gemini-2.5-pro",1.25,10.00,"frontier"),MP("google","text-embedding-004",0.006,0.0,"embedding"),
    MP("openai","gpt-4o-mini",0.15,0.60,"fast"),MP("openai","gpt-4o",2.50,10.00,"frontier"),
    MP("openai","gpt-4.1-mini",0.40,1.60,"balanced"),MP("anthropic","claude-sonnet-4",3.00,15.00,"frontier"),
    MP("anthropic","claude-haiku-3.5",0.80,4.00,"balanced")]

class CC:
    def __init__(self,db): self.db={p.model:p for p in db}
    def per_req(self,m,i,o):
        p=self.db[m]; ic=i*p.input_per_1m/1e6; oc=o*p.output_per_1m/1e6
        return {"model":m,"total":round(ic+oc,6),"inr":round((ic+oc)*84,4)}
    def monthly(self,m,d,i=500,o=300):
        r=self.per_req(m,i,o); mu=r["total"]*d*30
        return {"model":m,"per_req":r["total"],"mo_usd":round(mu,2),"mo_inr":round(mu*84,0)}
    def compare(self,models,i,o,d=5000):
        res=[self.monthly(m,d,i,o) for m in models]
        res.sort(key=lambda x:x["mo_usd"]); mx=res[-1]["mo_usd"]
        for r in res: r["save"]=round((1-r["mo_usd"]/mx)*100) if mx else 0
        return res
    def rag(self,m,q=50,c=3000,o=400):
        ec=q*0.006/1e6; lc=self.per_req(m,q+c,o)["total"]
        return {"total":round(ec+lc,6),"ctx_pct":round(c/(q+c)*100)}

c=CC(DB)
print("1. Per-Request Cost:")
for n,m,i,o in [("Q&A","gemini-2.0-flash",500,300),("RAG","gemini-2.0-flash",3500,400),
    ("Agent","gemini-2.0-flash",4000,1500),("Doc","gemini-2.5-pro",20000,2000)]:
    r=c.per_req(m,i,o); print(f"  {n:<12} {m:<22} ${r['total']:.6f} Rs{r['inr']:.4f}")

print(f"\n2. Monthly at 5K/day:")
for r in c.compare(["gemini-2.0-flash","gpt-4o-mini","gpt-4.1-mini","gpt-4o","claude-sonnet-4"],500,300,5000):
    print(f"  {r['model']:<22} ${r['mo_usd']:>8.2f}/mo Rs{r['mo_inr']:>8,.0f} {r['save']:>3}% cheaper")

print(f"\n3. RAG Hidden Cost:")
for m in ["gemini-2.0-flash","gemini-2.5-pro","claude-sonnet-4"]:
    r=c.rag(m); print(f"  {m:<22} ${r['total']:.6f} context={r['ctx_pct']}%")


</details>


---
## Exercise 2 (Medium): BigQuery Logging


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
from dataclasses import dataclass
import random, uuid
from datetime import datetime, timedelta
from collections import defaultdict
random.seed(42)

@dataclass
class MP:
    provider:str; model:str; input_per_1m:float; output_per_1m:float; category:str

DB=[MP("google","gemini-2.0-flash",0.10,0.40,"fast"),MP("google","gemini-2.5-pro",1.25,10.00,"frontier"),
    MP("openai","gpt-4o-mini",0.15,0.60,"fast"),MP("anthropic","claude-sonnet-4",3.00,15.00,"frontier")]

class CC:
    def __init__(self,db): self.db={p.model:p for p in db}
    def per_req(self,m,i,o):
        p=self.db[m]; ic=i*p.input_per_1m/1e6; oc=o*p.output_per_1m/1e6
        return {"model":m,"total":round(ic+oc,6)}

c=CC(DB)

class CL:
    def __init__(self,calc): self.calc=calc; self.logs=[]
    def log(self,m,i,o,lat,team="default",cached=False):
        r=self.calc.per_req(m,i,o)
        self.logs.append({"ts":(datetime.utcnow()-timedelta(hours=random.randint(0,168))).isoformat(),
            "id":uuid.uuid4().hex[:8],"team":team,"model":m,"provider":self.calc.db[m].provider,
            "in":i,"out":o,"tok":i+o,"cost":r["total"],"lat":lat,"cached":cached})

lg=CL(c)
teams=["product","research","support"]
mix=[("gemini-2.0-flash",0.5,(300,800),(100,400)),("gemini-2.5-pro",0.15,(2000,8000),(500,2000)),
    ("gpt-4o-mini",0.25,(300,600),(100,300)),("claude-sonnet-4",0.10,(1000,5000),(500,3000))]
for _ in range(100):
    r=random.random(); cum=0
    for m,w,ir,oR in mix:
        cum+=w
        if r<=cum: break
    lg.log(m,random.randint(*ir),random.randint(*oR),round(max(5,random.gauss(800 if "flash" in m or "mini" in m else 2500,300))),random.choice(teams),random.random()<0.3)

print("BigQuery Dashboard:")
bt=defaultdict(lambda:{"c":0,"$":0})
for l in lg.logs: bt[l["team"]]["c"]+=1; bt[l["team"]]["$"]+=l["cost"]
print("  By Team:")
for t,d in sorted(bt.items(),key=lambda x:-x[1]["$"]): print(f"    {t:<12} {d['c']:>4} calls ${d['$']:.4f}")

bm=defaultdict(lambda:{"c":0,"$":0})
for l in lg.logs: bm[l["model"]]["c"]+=1; bm[l["model"]]["$"]+=l["cost"]
print("  By Model:")
for m,d in sorted(bm.items(),key=lambda x:-x[1]["$"]): print(f"    {m:<22} {d['c']:>4} ${d['$']:.4f}")

cached=sum(1 for l in lg.logs if l["cached"])
saved=sum(l["cost"] for l in lg.logs if l["cached"])
print(f"  Cache: {cached}/100 ({cached}%) saved ${saved:.4f}")
outliers=sorted([l for l in lg.logs if l["cost"]>0.01],key=lambda x:-x["cost"])
print(f"  Outliers >$0.01: {len(outliers)}")
for o in outliers[:3]: print(f"    {o['id']} {o['team']:>8} {o['model']:<22} ${o['cost']:.6f}")


</details>


---
## Exercise 3 (Medium): Budget Alerting


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class BA:
    LEVELS=[(1.00,"BLOCKED"),(0.90,"CRITICAL"),(0.75,"WARNING"),(0.50,"INFO")]
    def __init__(self,budgets): self.b=budgets; self.s={t:0 for t in budgets}
    def spend(self,t,a): self.s[t]=self.s.get(t,0)+a
    def check(self,t):
        pct=self.s[t]/self.b[t]*100
        for th,lv in self.LEVELS:
            if self.s[t]>=self.b[t]*th: return lv,round(pct,1)
        return "OK",round(pct,1)

ba=BA({"product":50,"research":200,"support":100})
daily={"product":[6,7,8,9,7,8,10],"research":[25,30,28,35,32,40,38],"support":[12,14,13,15,16,14,18]}

print("Budget Alerting:")
print(f"  {'Day':>4} {'Team':<12} {'$Day':>6} {'$Total':>7} {'%':>5} {'Alert':>10}")
for d in range(7):
    for t in ba.b:
        ba.spend(t,daily[t][d]); lv,pct=ba.check(t)
        print(f"  {d+1:>4} {t:<12} ${daily[t][d]:>5} ${ba.s[t]:>6.0f} {pct:>4.0f}% {lv:>10}")

print(f"\n  Hourly Spikes ($5 threshold):")
hrs=[random.gauss(2,1) for _ in range(24)]; hrs[14]=8.5; hrs[22]=6.2
for h,cost in enumerate(hrs):
    if max(0,cost)>5: print(f"    {h:02d}:00 ${max(0,cost):.2f} ALERT")


</details>


---
## Exercise 4 (Challenge): Anomaly Detection


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class CAD:
    def __init__(self,th=3.0): self.th=th; self.hist=[]; self.anoms=[]
    def add(self,d,h,cost): self.hist.append({"d":d,"h":h,"c":cost})
    def avg(self,d,h,w=7):
        past=[x["c"] for x in self.hist if x["h"]==h and d-w<=x["d"]<d]
        return sum(past)/len(past) if past else 0
    def detect(self,d,h,cost):
        a=self.avg(d,h)
        if a>0 and cost>a*self.th:
            r={"d":d,"h":h,"cost":round(cost,2),"avg":round(a,2),"ratio":round(cost/a,1)}
            self.anoms.append(r); return r
        return None

det=CAD(3.0)
def hp(h):
    if 2<=h<=6: return 0.5
    if 9<=h<=17: return 3.0
    return 1.5

inj={}
for d in range(14):
    for h in range(24):
        cost=max(0.1,hp(h)+random.gauss(0,0.3))
        if d==10 and h==14: cost=15.0; inj[(10,14)]="cost_spike"
        elif d==11 and h==10: cost=12.0; inj[(11,10)]="token_bloat"
        elif d==12 and h==16: cost=18.0; inj[(12,16)]="model_switch"
        det.add(d,h,cost)
        if d>=7: det.detect(d,h,cost)

print("Anomaly Detection:")
print(f"  Injected: {len(inj)} | Detected: {len(det.anoms)}")
dk=set()
for a in det.anoms:
    k=(a["d"],a["h"]); dk.add(k)
    t=inj.get(k,"false_positive")
    print(f"  Day {a['d']:>2} Hr {a['h']:>2} ${a['cost']:>5.1f} avg=${a['avg']:>4.1f} {a['ratio']}x {t}")
tp=len(dk&set(inj.keys())); fp=len(dk-set(inj.keys())); ms=len(set(inj.keys())-dk)
print(f"  TP:{tp} FP:{fp} Missed:{ms} Precision:{tp/max(tp+fp,1)*100:.0f}% Recall:{tp/max(len(inj),1)*100:.0f}%")


</details>
